# Registering the released NIfTIs directly, with SimpleITK

A test of registering each patient's studies on the **source NIfTIs**, before any of the builder's
processing (no crop, no median/MAD normalisation, no heuristic masks), with SimpleITK's standard
pipeline:

1. **reference** -- the study with the most axial slices (the builder's rule)
2. **initialise** -- align the centres of mass (`CenteredTransformInitializer`, MOMENTS)
3. **rigid** -- 6 degrees of freedom
4. **affine** -- 12, started from the rigid result; absorbs scale, including the release's
   unmeasured "1 mm" voxel size
5. both stages: Mattes mutual information, a 3-level pyramid (shrink 4 / 2 / 1), regular-step
   gradient descent with physically scaled steps, the metric sampled inside the reference's head
   mask

The four contrasts of a session were co-registered by the release itself, so ONE transform per
study, fitted on T1, moves all of them. Nothing is written to the dataset; transforms are saved only
if `SAVE_DIR` is set.

Needs `pip install SimpleITK` in the environment.

## Knobs

In [ ]:
import os, sys, glob, time, math
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) != "notebooks":
    REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
from preprocessing.nyumets_h5 import RULES, SKIP_RE          # the builder's file-naming rules

NIFTI_ROOT = globals().get("PATIENT_ROOT") or next(
    (p for p in (os.path.join(REPO_ROOT, "..", "datasets", "NYUMets", "data", "imaging", "patientId"),
                 "/gpfs/data/fenglab/LiFeng/NYUMets/NYUMets/data/imaging/patientId")
     if os.path.isdir(p)), None)
assert NIFTI_ROOT, "set NIFTI_ROOT to the NIfTI patientId level"

# patients with known problems on the centroid build, from the earlier comparisons
PATIENTS = ["21755694", "25260276", "23955252", "10178949"]
OTHERS_PER_PATIENT = 2         # other studies registered per patient
SCALES = {"T1": 2500.0, "T1ce": 2390.0}          # the NYUMets display scales
VMIN, VMAX = 0.0, 1.0

# registration
BINS = 50                      # Mattes mutual information histogram bins
SAMPLING = 0.10                # fraction of masked voxels sampled per iteration
SHRINK, SMOOTH = [4, 2, 1], [2.0, 1.0, 0.0]      # pyramid; smoothing sigmas in mm
ITERS = 300                    # per level (a cap: the optimiser stops when its step is tiny)
DO_AFFINE = True               # rigid only if False
SAVE_DIR = None                # e.g. "cache/sitk_tfm": write each transform as <patient>_<session>.tfm

print("SimpleITK", sitk.Version_VersionString(), "| NIfTIs under", NIFTI_ROOT)

## Helpers

`read` reorients every volume to LPS index order, so `GetArrayFromImage(img)[z]` is an axial slice
already in radiological display convention (anterior up, patient's right on the image's left).
Reorienting changes only the index order, never where the voxels are in space, so registration is
unaffected.

In [ ]:
def sessions_of(pid):
    """{session: {contrast: path}} with at least T1 and T1ce -- the builder's naming rules."""
    out = defaultdict(dict)
    for f in glob.glob(os.path.join(NIFTI_ROOT, pid, "**", "*.nii*"), recursive=True):
        b = os.path.basename(f)
        if SKIP_RE.search(b):
            continue
        lab = next((l for l, rx in RULES if rx.search(b)), None)
        if lab:
            out[os.path.basename(os.path.dirname(f)) or "single"].setdefault(lab, f)
    return {s: d for s, d in sorted(out.items()) if "T1" in d and "T1ce" in d}


def read(path):
    return sitk.DICOMOrient(sitk.ReadImage(path, sitk.sitkFloat32), "LPS")


def head_mask(img):
    """Foreground by Otsu, closed and hole-filled. The released volumes are skull-stripped, so
    this is essentially the brain."""
    m = sitk.OtsuThreshold(img, 0, 1, 128)
    m = sitk.BinaryMorphologicalClosing(m, [2, 2, 2])
    return sitk.BinaryFillhole(m)


def _method(fmask):
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=BINS)
    R.SetMetricSamplingStrategy(R.RANDOM)
    R.SetMetricSamplingPercentage(SAMPLING, 0)            # fixed seed: reproducible
    R.SetMetricFixedMask(fmask)
    R.SetInterpolator(sitk.sitkLinear)
    # Regular-step gradient descent: the step halves whenever the direction reverses, and the
    # level ends when the step is too small to matter. Plain gradient descent with a fixed rate
    # stopped after ~30 iterations on the synthetic test and could end WORSE than it started.
    R.SetOptimizerAsRegularStepGradientDescent(learningRate=2.0, minStep=1e-4,
                                               numberOfIterations=ITERS, relaxationFactor=0.5,
                                               gradientMagnitudeTolerance=1e-8)
    R.SetOptimizerScalesFromPhysicalShift()
    R.SetShrinkFactorsPerLevel(SHRINK)
    R.SetSmoothingSigmasPerLevel(SMOOTH)
    R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    return R


def register(fixed, moving, fmask):
    """-> {"rigid": Euler3DTransform, "affine": AffineTransform or None, "secs": ...}.
    Each transform maps a REFERENCE point to the moving image's point (SimpleITK's convention),
    which is what sitk.Resample wants."""
    t0 = time.time()
    rigid = sitk.Euler3DTransform(sitk.CenteredTransformInitializer(
        fixed, moving, sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.MOMENTS))
    R = _method(fmask)
    R.SetInitialTransform(rigid, inPlace=True)
    R.Execute(fixed, moving)
    out = {"rigid": rigid, "affine": None, "rigid_stop": R.GetOptimizerStopConditionDescription()}
    if DO_AFFINE:
        aff = sitk.AffineTransform(3)
        aff.SetCenter(rigid.GetCenter())
        aff.SetMatrix(rigid.GetMatrix())
        aff.SetTranslation(rigid.GetTranslation())
        R = _method(fmask)
        R.SetInitialTransform(aff, inPlace=True)
        R.Execute(fixed, moving)
        out["affine"] = aff
    out["secs"] = time.time() - t0
    return out


def to_ref(img, ref, tx, nearest=False):
    """`img` resampled onto `ref`'s grid through `tx` (identity = by the headers alone)."""
    interp = sitk.sitkNearestNeighbor if nearest else sitk.sitkLinear
    return sitk.GetArrayFromImage(sitk.Resample(img, ref, tx, interp, 0.0, img.GetPixelID()))


def ncc(a, b, m):
    if int(m.sum()) < 500:
        return float("nan")
    x, y = a[m] - a[m].mean(), b[m] - b[m].mean()
    return float((x * y).sum() / max(math.sqrt(float((x * x).sum() * (y * y).sum())), 1e-8))


def describe(tx):
    """Rotation angle (deg), the three scales, and |translation| (mm) of a 3x3 + shift."""
    A = np.asarray(tx.GetMatrix()).reshape(3, 3)
    U, s, Vt = np.linalg.svd(A)
    R = U @ Vt
    ang = math.degrees(math.acos(min(1.0, max(-1.0, (np.trace(R) - 1) / 2))))
    return ang, tuple(sorted(s)), float(np.linalg.norm(tx.GetTranslation()))

## Register

For each patient, the deepest study is the reference and up to `OTHERS_PER_PATIENT` others are
registered onto it. Each row of the table scores the other study's T1 (and CT1, moved by the SAME
transform) against the reference inside the reference's head mask, three ways:

* `header` -- resampled by the NIfTI headers alone: what the files say before any registration
* `rigid` -- after the rigid stage
* `affine` -- after the affine stage

`rot`, `scales` and `|t|` describe the final transform: rotation in degrees, the three scale factors
(1 = none) and the translation in mm. Expect one study per patient to take a minute or so on CPU.

In [ ]:
RESULTS = []
for pid in PATIENTS:
    ses = sessions_of(pid)
    if len(ses) < 2:
        print("%s: %d usable session(s) under %s -- skipped" % (pid, len(ses), NIFTI_ROOT))
        continue
    t1 = {s: read(d["T1"]) for s, d in ses.items()}
    ref = max(t1, key=lambda s: (t1[s].GetSize()[2], s))        # the deepest study
    fixed = t1[ref]
    fmask = head_mask(fixed)
    fm = sitk.GetArrayFromImage(fmask).astype(bool)
    fixed_ct1 = read(ses[ref]["T1ce"])
    A1, C1 = sitk.GetArrayFromImage(fixed), sitk.GetArrayFromImage(fixed_ct1)
    print("%s: reference %s  size %s  spacing %s" % (pid, ref, fixed.GetSize(),
                                                  tuple(round(v, 3) for v in fixed.GetSpacing())))
    for oth in [s for s in sorted(ses) if s != ref][:OTHERS_PER_PATIENT]:
        moving, moving_ct1 = t1[oth], read(ses[oth]["T1ce"])
        mmask = head_mask(moving)
        reg = register(fixed, moving, fmask)
        final = reg["affine"] or reg["rigid"]
        txs = {"header": sitk.Transform(3, sitk.sitkIdentity), "rigid": reg["rigid"]}
        if reg["affine"] is not None:
            txs["affine"] = reg["affine"]
        row = dict(patient=pid, ref=ref, other=oth, secs=reg["secs"],
                   size=moving.GetSize(), spacing=moving.GetSpacing(), stop=reg["rigid_stop"],
                   tx=final, A1=A1, C1=C1, fm=fm, moved={})
        for name, tx in txs.items():
            b1, bc = to_ref(moving, fixed, tx), to_ref(moving_ct1, fixed, tx)
            bm = to_ref(mmask, fixed, tx, nearest=True).astype(bool)
            row["ncc_" + name] = ncc(A1, b1, fm & bm)
            row["ncc_ct1_" + name] = ncc(C1, bc, fm & bm)
            row["moved"][name] = (b1, bc, bm)
        row["rot"], row["scales"], row["shift"] = describe(final)
        if SAVE_DIR:
            os.makedirs(SAVE_DIR, exist_ok=True)
            sitk.WriteTransform(final, os.path.join(SAVE_DIR, "%s_%s.tfm" % (pid, oth)))
        RESULTS.append(row)
        print("   %-12s size %-16s T1 ncc  header %.3f  rigid %.3f  affine %s   (%.0fs)"
              % (oth, str(moving.GetSize()), row["ncc_header"], row["ncc_rigid"],
                 ("%.3f" % row["ncc_affine"]) if "ncc_affine" in row else "-", reg["secs"]))

print("\n%-10s %-12s %8s %8s %8s %8s %8s %6s %-20s %7s"
      % ("patient", "other", "T1 head", "T1 rig", "T1 aff", "CT1 hd", "CT1 aff", "rot", "scales", "|t| mm"))
for r in RESULTS:
    print("%-10s %-12s %8.3f %8.3f %8.3f %8.3f %8.3f %6.1f %-20s %7.1f"
          % (r["patient"], r["other"], r["ncc_header"], r["ncc_rigid"],
             r.get("ncc_affine", float("nan")), r["ncc_ct1_header"],
             r.get("ncc_ct1_affine", r["ncc_ct1_rigid"]), r["rot"],
             " ".join("%.3f" % v for v in r["scales"]), r["shift"]))

## Look at it

One figure per registered study. Axial rows at three levels of the reference's brain, T1 and CT1,
then one SAGITTAL row: a tilt between visits shows up there as structures that line up at one end
of the head and not the other, which no axial view makes obvious. Columns: the reference, the other
study by the headers alone, after rigid, after affine, and |reference - other| before (headers) and
after (final transform), inside the reference's mask.

In [ ]:
LEVELS = (0.3, 0.5, 0.7)       # fractions through the reference brain's axial extent
SHOW = None                    # e.g. [0, 2]: which rows of RESULTS to draw (None = all)


def figure(r):
    names = [n for n in ("header", "rigid", "affine") if n in r["moved"]]
    last = names[-1]
    zs = np.flatnonzero(r["fm"].reshape(r["fm"].shape[0], -1).any(1))
    picks = [int(zs[0] + f * (zs[-1] - zs[0])) for f in LEVELS]
    x_mid = int(np.flatnonzero(r["fm"].any((0, 1))).mean())
    rows, labels = [], []
    for z in picks:
        for ci, name in ((0, "T1"), (1, "CT1")):
            ref = (r["A1"] if ci == 0 else r["C1"])[z] / SCALES["T1" if ci == 0 else "T1ce"]
            mov = [r["moved"][n][ci][z] / SCALES["T1" if ci == 0 else "T1ce"] for n in names]
            rows.append([ref] + mov + [np.abs(ref - mov[0]) * r["fm"][z],
                                       np.abs(ref - mov[-1]) * r["fm"][z]])
            labels.append("%s  axial %d" % (name, z))
    sag = lambda v: np.flipud(v[:, :, x_mid])           # superior up
    ref = sag(r["A1"]) / SCALES["T1"]
    mov = [sag(r["moved"][n][0]) / SCALES["T1"] for n in names]
    msk = sag(r["fm"])
    rows.append([ref] + mov + [np.abs(ref - mov[0]) * msk, np.abs(ref - mov[-1]) * msk])
    labels.append("T1  sagittal")

    cols = ["reference\n" + r["ref"]] + ["other, %s\n%s" % (n, r["other"]) for n in names] \
        + ["|ref - other|\nheader", "|ref - other|\n" + last]
    fig, ax = plt.subplots(len(rows), len(cols), figsize=(2.3 * len(cols), 2.3 * len(rows)),
                           squeeze=False)
    for i, row in enumerate(rows):
        for j, img in enumerate(row):
            diff = j >= len(cols) - 2
            ax[i, j].imshow(img, cmap="magma" if diff else "gray", vmin=0.0,
                            vmax=0.5 * VMAX if diff else VMAX, aspect="auto" if i == len(rows) - 1 else "equal")
            ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
            if i == 0:
                ax[i, j].set_title(cols[j], fontsize=8)
        ax[i, 0].set_ylabel(labels[i], fontsize=8)
    fig.suptitle("%s: %s onto %s   T1 ncc header %.3f -> %s %.3f   (rot %.1f deg, scales %s)"
                 % (r["patient"], r["other"], r["ref"], r["ncc_header"], last, r["ncc_" + last],
                    r["rot"], " ".join("%.3f" % v for v in r["scales"])), fontsize=9)
    plt.tight_layout(); plt.show()


for k, r in enumerate(RESULTS):
    if SHOW is None or k in SHOW:
        figure(r)

## Along the head

Per reference axial slice, the correlation of the other study's T1 with the reference: headers alone
against the final transform. A flat, high curve means the alignment holds over the whole head. A
curve that is good in the middle and falls off toward one end is a residual tilt or scale error
along z -- the failure the centroid build had, and the thing an affine fit should remove.

In [ ]:
fig, axes = plt.subplots(1, max(len(RESULTS), 1), figsize=(3.6 * max(len(RESULTS), 1), 2.8),
                         squeeze=False)
for k, r in enumerate(RESULTS):
    ax = axes[0, k]
    last = "affine" if "affine" in r["moved"] else "rigid"
    for name, style in (("header", ":"), (last, "-")):
        b1, _, bm = r["moved"][name]
        zs, vals = [], []
        for z in range(r["A1"].shape[0]):
            m = r["fm"][z] & bm[z]
            if m.sum() > 300:
                zs.append(z); vals.append(ncc(r["A1"][z], b1[z], m))
        ax.plot(zs, vals, style, label=name)
    ax.set_ylim(-0.2, 1.0); ax.grid(alpha=0.3)
    ax.set_title("%s / %s" % (r["patient"], r["other"]), fontsize=9)
    ax.set_xlabel("reference axial slice")
axes[0, 0].set_ylabel("T1 correlation"); axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

# SimpleITK against the `fit` build, on the same patients

Both results are scored in the **`fit` build's own space**: the same reference image, the same
`slice_index`, the same pixels. For each (patient, other study) the `fit` build holds:

* SimpleITK registers the other study's T1 NIfTI onto the reference T1 NIfTI -- using the
  reference the `fit` build used, so both answer the same question;
* the other study's T1, CT1 and head mask are resampled onto the reference NIfTI's grid through
  that transform, then put through the builder's own reorientation (`as_closest_canonical`) and
  `--crop` (`center_crop_or_pad`), so they line up index for index with the h5s.

`GEOMETRY CHECK` confirms that mapping first: the reference NIfTI pushed through it must reproduce
the `fit` build's stored reference exactly. If it does not, nothing below is comparable.

Scored per slice, inside the INTERSECTION of the reference's brain mask and both methods' masks for
the other study, so both see the identical pixels:

* `ncc` -- masked correlation with the reference, T1 and CT1
* the diagnostic's residual search (`scripts/diagnose_nyumets_registration.py`): the extra in-plane
  rotation / scale / shift that would still improve each result, and the slice offset that matches
  best. Small residuals = nothing left to fix.

In [ ]:
import h5py, torch, nibabel as nib
from preprocessing.nyumets_h5 import center_crop_or_pad
from scripts.diagnose_nyumets_registration import similarity_search, best_dz
from visualization.image import set_display_orient, subplot_images

FIT_ROOT = "~/scratch/datasets/NYUMets_h5_fit_test"
CMP_PATIENTS = None            # e.g. ["21755694"]; None = draw from the fit build
CMP_N_PATIENTS = 6             # patients drawn when CMP_PATIENTS is None
CMP_OTHERS = 2                 # other studies per patient
CMP_SLICES = 6                 # slices per pair scored by the residual search
CMP_RANGE = (50, 90)           # original slice indices compared (what training uses)
CMP_SEED = 0
CMP_SHOW = 3                   # pairs drawn as figures
ANGLES = np.arange(-15, 15.01, 1.0)
SCALE_GRID = np.round(np.arange(0.88, 1.1201, 0.02), 4)

FIT_ROOT = os.path.abspath(os.path.expanduser(FIT_ROOT))
set_display_orient("radiological")          # the h5 axes are canonical RAS; display only


def fit_index(root):
    """{patient: {session: h5 path}} of a built set."""
    out = {}
    for p in sorted(glob.glob(os.path.join(root, "*", "*_img.h5"))):
        with h5py.File(p, "r") as f:
            pid, ses = f.attrs.get("patient"), f.attrs.get("session")
        pid = pid.decode() if isinstance(pid, bytes) else str(pid)
        ses = ses.decode() if isinstance(ses, bytes) else str(ses)
        out.setdefault(pid, {})[ses] = p
    return out


def h5_load(path):
    with h5py.File(path, "r") as f:
        d = dict(z=np.asarray(f["slice_index"]).astype(int),
                 t1=np.asarray(f["img_raw"][..., 1], dtype=np.float32),
                 ct1=np.asarray(f["img_raw"][..., 2], dtype=np.float32),
                 m=np.asarray(f["mask"][..., 0]).astype(bool),
                 crop=int(f.attrs.get("crop_size", -1)),
                 ref=str(f.attrs.get("reg_reference", "")), method=str(f.attrs.get("reg_method", "")))
    return d


def builder_space(arr_kji, ref_path, crop, mask=False):
    """An array on the reference NIfTI's OWN grid (SimpleITK order k, j, i) -> the builder's
    (D, H, W): canonical RAS via nibabel, as load_ras does, then the in-plane --crop."""
    ref = nib.load(ref_path)
    img = nib.Nifti1Image(np.ascontiguousarray(np.asarray(arr_kji).transpose(2, 1, 0)), ref.affine)
    v = np.asarray(nib.as_closest_canonical(img).dataobj)
    while v.ndim > 3:
        v = v[..., 0]
    t = torch.from_numpy(np.ascontiguousarray(v.transpose(2, 0, 1).astype(np.float32)))[:, None]
    if crop and crop > 0:
        t = center_crop_or_pad(t, crop, h_axis=2, w_axis=3)
    out = t[:, 0].numpy()
    return out > 0.5 if mask else out


FIT = fit_index(FIT_ROOT)
multi = [p for p in sorted(FIT) if len(FIT[p]) > 1]
rng_c = np.random.default_rng(CMP_SEED)
pids = [str(p) for p in CMP_PATIENTS] if CMP_PATIENTS else \
    [multi[i] for i in sorted(rng_c.permutation(len(multi))[:CMP_N_PATIENTS])]
print("fit build: %s  (%d patients, %d with >1 study); comparing %d patient(s)"
      % (FIT_ROOT, len(FIT), len(multi), len(pids)))

In [ ]:
CMP = []
for pid in pids:
    if pid not in FIT or len(FIT[pid]) < 2:
        print(pid + ": not a multi-study patient of the fit build -- skipped")
        continue
    nif = sessions_of(pid)
    first = h5_load(next(iter(FIT[pid].values())))
    ref = first["ref"][len(pid) + 1:] if first["ref"].startswith(pid + "_") else None
    if ref not in FIT[pid] or ref not in nif:
        print(pid + ": the fit build's reference " + repr(first["ref"]) + " has no NIfTI -- skipped")
        continue
    R = h5_load(FIT[pid][ref])
    crop = R["crop"]
    fixed_native = sitk.ReadImage(nif[ref]["T1"], sitk.sitkFloat32)
    fixed_lps = sitk.DICOMOrient(fixed_native, "LPS")
    fmask_lps = head_mask(fixed_lps)

    # ---- GEOMETRY CHECK: the reference NIfTI through the mapping == the fit build's reference --
    ref_bs = builder_space(sitk.GetArrayFromImage(fixed_native), nif[ref]["T1"], crop)
    rows_ok = R["z"] < ref_bs.shape[0]
    err = np.abs(ref_bs[R["z"][rows_ok]] - R["t1"][rows_ok]).max() / max(np.abs(R["t1"]).max(), 1e-6)
    print("%s: reference %s   GEOMETRY CHECK %s (max rel. diff %.1e)"
          % (pid, ref, "ok" if err < 1e-4 else "FAILED", err))
    if err >= 1e-4:
        print("   the mapping into the fit build's space is wrong for this patient -- skipped")
        continue

    for oth in [s for s in sorted(FIT[pid]) if s != ref and s in nif][:CMP_OTHERS]:
        O = h5_load(FIT[pid][oth])
        mov = {c: sitk.ReadImage(nif[oth][c], sitk.sitkFloat32) for c in ("T1", "T1ce")}
        reg = register(fixed_lps, sitk.DICOMOrient(mov["T1"], "LPS"), fmask_lps)
        tx = reg["affine"] or reg["rigid"]
        moved = lambda img, interp=sitk.sitkLinear: sitk.GetArrayFromImage(
            sitk.Resample(img, fixed_native, tx, interp, 0.0, img.GetPixelID()))
        S = dict(t1=builder_space(moved(mov["T1"]), nif[ref]["T1"], crop),
                 ct1=builder_space(moved(mov["T1ce"]), nif[ref]["T1"], crop),
                 m=builder_space(moved(head_mask(mov["T1"]), sitk.sitkNearestNeighbor),
                                 nif[ref]["T1"], crop, mask=True))
        # the slices both builds hold, in range; fit rows indexed by its own slice_index
        zs = [int(z) for z in R["z"] if z in set(O["z"].tolist()) and z < S["t1"].shape[0]
              and CMP_RANGE[0] <= z < CMP_RANGE[1]]
        if not zs:
            print("   %s: no shared slice in %s" % (oth, CMP_RANGE)); continue
        rr = {int(z): k for k, z in enumerate(R["z"])}
        ro = {int(z): k for k, z in enumerate(O["z"])}
        recs = []
        for z in zs:
            mref = R["m"][rr[z]]
            joint = mref & O["m"][ro[z]] & S["m"][z]
            recs.append(dict(z=z,
                             fit_t1=ncc(R["t1"][rr[z]], O["t1"][ro[z]], joint),
                             sitk_t1=ncc(R["t1"][rr[z]], S["t1"][z], joint),
                             fit_ct1=ncc(R["ct1"][rr[z]], O["ct1"][ro[z]], joint),
                             sitk_ct1=ncc(R["ct1"][rr[z]], S["ct1"][z], joint)))
        # residual search on a spread of slices, each method with its own mask (as the diagnostic)
        pick = [zs[i] for i in np.linspace(0, len(zs) - 1, min(CMP_SLICES, len(zs))).astype(int)]
        T = lambda x: torch.from_numpy(np.ascontiguousarray(x))
        O_rows = {"fit": (T(O["t1"]), T(O["m"]), lambda z: ro[z]),
                  "sitk": (T(S["t1"]), T(S["m"]), lambda z: z)}
        resid = {"fit": [], "sitk": []}
        for z in pick:
            a, ma = T(R["t1"][rr[z]]), T(R["m"][rr[z]])
            for name, (vol, vm, row) in O_rows.items():
                j = row(z)
                r = similarity_search(a, ma, vol[j], vm[j], ANGLES, SCALE_GRID)
                r["dz"] = best_dz(a, ma, vol, vm, j, 8)
                resid[name].append(r)
        CMP.append(dict(pid=pid, ref=ref, oth=oth, recs=recs, resid=resid, R=R, O=O, S=S,
                        rr=rr, ro=ro, secs=reg["secs"], fit_method=O["method"]))
        md_ = lambda k: np.nanmedian([r[k] for r in recs])
        print("   %-12s %2d slices   T1 ncc  fit %.3f  sitk %.3f   CT1  fit %.3f  sitk %.3f   (%.0fs)"
              % (oth, len(recs), md_("fit_t1"), md_("sitk_t1"), md_("fit_ct1"), md_("sitk_ct1"),
                 reg["secs"]))

## Summary

Paired over every compared slice. `sitk better` counts the slices where SimpleITK's correlation
beats the fit build's on the identical pixels. The residual rows are the diagnostic's: the extra
in-plane fit and slice offset still available after each method.

In [ ]:
allr = [r for c in CMP for r in c["recs"]]
if not allr:
    print("nothing compared")
else:
    A = {k: np.array([r[k] for r in allr], dtype=float) for k in ("fit_t1", "sitk_t1", "fit_ct1", "sitk_ct1")}
    ok = np.isfinite(A["fit_t1"]) & np.isfinite(A["sitk_t1"])
    print("%d slices over %d pairs (%d patients)\n" % (ok.sum(), len(CMP), len({c["pid"] for c in CMP})))
    print("%-30s %10s %10s" % ("", "fit", "SimpleITK"))
    for name, k in (("T1 ncc, median", "t1"), ("CT1 ncc, median", "ct1")):
        print("%-30s %10.3f %10.3f" % (name, np.median(A["fit_" + k][ok]), np.median(A["sitk_" + k][ok])))
    for name, k in (("T1", "t1"), ("CT1", "ct1")):
        d = A["sitk_" + k][ok] - A["fit_" + k][ok]
        print("%-30s %10s %10.0f%%   (median difference %+.3f)"
              % (name + ": sitk better on", "", 100 * (d > 0).mean(), np.median(d)))
    R_ = {m: [r for c in CMP for r in c["resid"][m]] for m in ("fit", "sitk")}
    rows = [("residual |rotation| >= 3 deg", lambda r: abs(r["angle"]) >= 3),
            ("residual |scale - 1| >= 0.04", lambda r: abs(r["scale"] - 1) >= 0.04),
            ("residual shift >= 4 px", lambda r: math.hypot(r["dh"], r["dw"]) >= 4),
            ("best slice not within 1", lambda r: abs(r["dz"]) > 1),
            ("extra fit raises ncc >= 0.10", lambda r: r["after"] - r["before"] >= 0.10)]
    print()
    for name, f in rows:
        print("%-30s %9.0f%% %9.0f%%" % (name, *(100 * np.mean([f(r) for r in R_[m]]) for m in ("fit", "sitk"))))
    print("%-30s %10d %10d" % ("(slices searched)", len(R_["fit"]), len(R_["sitk"])))

    print("\n%-10s %-12s %8s %8s %8s %8s %6s" % ("patient", "other", "fit T1", "sitk T1", "fit CT1", "sitk CT1", "secs"))
    for c in CMP:
        m = lambda k: np.nanmedian([r[k] for r in c["recs"]])
        print("%-10s %-12s %8.3f %8.3f %8.3f %8.3f %6.0f"
              % (c["pid"], c["oth"], m("fit_t1"), m("sitk_t1"), m("fit_ct1"), m("sitk_ct1"), c["secs"]))

    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    for k, (name, key) in enumerate((("T1", "t1"), ("CT1", "ct1"))):
        ax[k].scatter(A["fit_" + key][ok], A["sitk_" + key][ok], s=8, alpha=0.6)
        ax[k].plot([-0.2, 1], [-0.2, 1], "k--", lw=0.8)
        ax[k].set_xlabel("correlation with reference, fit build")
        ax[k].set_ylabel("correlation with reference, SimpleITK")
        ax[k].set_title(name + ": above the line = SimpleITK better")
        ax[k].set_xlim(-0.2, 1); ax[k].set_ylim(-0.2, 1); ax[k].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## Side by side

The reference, the other study as the `fit` build stored it, the same study through SimpleITK, and
|reference - other| for each (one scale, inside the reference's mask). `img_raw` over the NYUMets
scales on the fixed 0..1 window; radiological display.

In [ ]:
for c in CMP[:CMP_SHOW]:
    z_all = [r["z"] for r in c["recs"]]
    zs = [z_all[i] for i in np.linspace(0, len(z_all) - 1, min(2, len(z_all))).astype(int)]
    rows, labels = [], []
    for z in zs:
        mref = c["R"]["m"][c["rr"][z]]
        for key, name, sc in (("t1", "T1", SCALES["T1"]), ("ct1", "CT1", SCALES["T1ce"])):
            ref = c["R"][key][c["rr"][z]] / sc
            fit = c["O"][key][c["ro"][z]] / sc
            sit = c["S"][key][z] / sc
            rows.append([torch.from_numpy(x) for x in
                         (ref, fit, sit, np.abs(ref - fit) * mref, np.abs(ref - sit) * mref)])
            rec = next(r for r in c["recs"] if r["z"] == z)
            labels.append("%s  z=%d\nfit %.2f | sitk %.2f" % (name, z, rec["fit_" + key], rec["sitk_" + key]))
    subplot_images(rows, row_labels=labels,
                   col_titles=["reference\n" + c["ref"], "other, fit build\n" + c["oth"],
                               "other, SimpleITK\n" + c["oth"], "|ref - other|\nfit", "|ref - other|\nSimpleITK"],
                   cmap=["gray", "gray", "gray", "magma", "magma"],
                   vmin=[VMIN, VMIN, VMIN, 0.0, 0.0], vmax=[VMAX, VMAX, VMAX, 0.5 * VMAX, 0.5 * VMAX],
                   magnitude=False, panel_size=(2.6, 2.9),
                   suptitle="%s: %s onto %s" % (c["pid"], c["oth"], c["ref"]), show=False)
    plt.show()